In [1]:
import os
import ot
import gc
import k3d
import torch
import trimesh
import warnings
from tqdm import *
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
from tqdm import tqdm
import seaborn as sns
from pathlib import Path
import scipy.sparse as sp
import matplotlib.cm as cm
from anndata import AnnData
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from numpy.random import RandomState
import matplotlib.font_manager as fm
from matplotlib.gridspec import GridSpec
from sklearn.metrics import jaccard_score
from scipy.stats import fisher_exact, norm
from sklearn.neighbors import NearestNeighbors
from typing import Literal, Optional, Tuple, Union
from matplotlib.colors import ListedColormap, rgb2hex
from sklearn.metrics.pairwise import euclidean_distances
from matplotlib.font_manager import fontManager, FontProperties
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar

import marsilea as ma
import re
from matplotlib_scalebar.scalebar import ScaleBar

plt.rcParams['pdf.fonttype'] = 42
import matplotlib.colors as mcolors
from matplotlib.font_manager import fontManager, FontProperties
import os
fontManager.addfont('/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial.ttf')
font = FontProperties(fname='/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial.ttf')
font_name = font.get_name()
plt.rcParams['font.family'] = font_name


tick_font = FontProperties(fname='/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/Arial-ItalicMT.otf', style = 'italic')

In [2]:
adata_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/merfish_mouseBrain_concat_embeddings.h5ad"
csv_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/example/05_merfish_mouseBrain/cluster_to_cluster_annotation_membership.csv"
json_path = "/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/rapids_analysis/author_colormaps/author_colormap.json"


adata = sc.read_h5ad(adata_path)
df = pd.read_csv(csv_path)

def strip_author_id(x):
    return re.sub(r"^\s*\d+\s+", "", str(x)).strip()

class_df = df[df["cluster_annotation_term_set_name"] == "class"].copy()
class_df["cluster_alias"] = class_df["cluster_alias"].astype(str)
class_df["author_class"] = class_df["cluster_annotation_term_name"].map(strip_author_id)

cluster_to_class = (
    class_df
    .drop_duplicates("cluster_alias")
    .set_index("cluster_alias")["author_class"]
    .to_dict()
)

adata.obs["author_class"] = (
    adata.obs["cluster_id_transfer"]
    .astype(str)
    .map(cluster_to_class)
    .astype("category")
)

In [3]:
import json

with open(json_path, "r", encoding="utf-8") as f:
    author_cmap = json.load(f)

print(author_cmap.keys())

class_palette = author_cmap["author_term_set_palettes"]["class"]
cluster_palette = author_cmap["obs_key_palettes"]["cluster_id_transfer"]
subclass_palette = author_cmap["obs_key_palettes"]["subclass_transfer"]


donor_id_colormap = {
    'C57BL6J-1': '#73BBF4', 
    'C57BL6J-2': '#284D76', 
    'C57BL6J-3': '#DF95D5', 
    'C57BL6J-4': '#791E25',
}

dict_keys(['source_membership', 'source_h5ad', 'obs_key_palettes', 'author_term_set_palettes'])


In [4]:
author_cmap['obs_key_palettes'].keys()

dict_keys(['cluster_id_transfer', 'subclass_transfer'])

In [5]:
import numpy as np
import pandas as pd
from matplotlib.colors import to_rgb, to_hex
import colorsys


def clamp(x, lo=0.0, hi=1.0):
    return max(lo, min(hi, x))


def tweak_color(
    hex_color,
    lighten=0.0,
    hue_shift=0.0,
    sat_scale=1.0,
):
    """
    在保留原始色系的基础上，调整：
    lighten: 向白色混合，0 原色，1 白色
    hue_shift: 色相偏移，建议小范围，如 -0.06 到 0.06
    sat_scale: 饱和度缩放，>1 更鲜艳，<1 更灰
    """
    r, g, b = to_rgb(hex_color)

    h, l, s = colorsys.rgb_to_hls(r, g, b)

    h = (h + hue_shift) % 1.0
    s = clamp(s * sat_scale)

    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)

    rgb = np.array([r2, g2, b2])
    white = np.array([1, 1, 1])

    new_rgb = rgb * (1 - lighten) + white * lighten
    return to_hex(new_rgb)


def symmetric_offsets(n, max_shift):
    """
    生成类似：
    0, +1, -1, +2, -2 ...
    这样最重要的 cluster 最接近原色，后面的逐渐偏移。
    """
    if n == 1 or max_shift == 0:
        return [0.0] * n

    offsets = [0.0]
    step = max_shift / max(1, np.ceil((n - 1) / 2))

    k = 1
    while len(offsets) < n:
        offsets.append(k * step)
        if len(offsets) < n:
            offsets.append(-k * step)
        k += 1

    return offsets[:n]


def build_leiden_palette_from_author(
    adata,
    leiden_key="cell_leiden",
    author_key="author_class",
    class_palette=None,
    min_lighten=0.0,
    max_lighten=0.45,
    max_hue_shift=0.07,
    min_sat_scale=0.75,
    max_sat_scale=1.15,
):
    obs = adata.obs[[leiden_key, author_key]].dropna().copy()

    obs[leiden_key] = obs[leiden_key].astype(str)
    obs[author_key] = obs[author_key].astype(str)

    confusion = pd.crosstab(
        obs[leiden_key],
        obs[author_key],
        normalize="index"
    )

    best_class = confusion.idxmax(axis=1)
    best_score = confusion.max(axis=1)

    mapping_df = pd.DataFrame({
        "cell_leiden": confusion.index,
        "matched_author_class": best_class.values,
        "matched_fraction": best_score.values,
    })

    leiden_palette = {}

    for author_class, sub_df in mapping_df.groupby("matched_author_class"):
        if class_palette is None or author_class not in class_palette:
            base_color = "#808080"
        else:
            base_color = class_palette[author_class]

        # 匹配度最高的 Leiden 最接近 author_class 原色
        sub_df = sub_df.sort_values("matched_fraction", ascending=False)

        n = len(sub_df)
        hue_offsets = symmetric_offsets(n, max_hue_shift)

        for rank, (_, row) in enumerate(sub_df.iterrows()):
            leiden = row["cell_leiden"]

            if n == 1:
                lighten = min_lighten
                sat_scale = 1.0
            else:
                t = rank / (n - 1)

                # 亮度逐渐增加
                lighten = min_lighten + (max_lighten - min_lighten) * t

                # 饱和度做轻微变化：前面的更鲜明，后面的略灰一些
                sat_scale = max_sat_scale + (min_sat_scale - max_sat_scale) * t

            leiden_palette[leiden] = tweak_color(
                base_color,
                lighten=lighten,
                hue_shift=hue_offsets[rank],
                sat_scale=sat_scale,
            )

    return leiden_palette, mapping_df, confusion

In [6]:
import re
import numpy as np
import pandas as pd

try:
    from scipy.stats import fisher_exact
except Exception:
    fisher_exact = None


def auto_find_enriched_groups(
    adata,
    group_key,
    target_key,
    target_regex=None,
    target_values=None,
    section_slices=None,
    section_key="brain_section_label",
    min_group_cells=30,
    min_target_cells=10,
    min_enrichment=2.0,
    min_frac_group_in_target=0.20,
    max_groups=30,
    use_fdr=True,
    verbose=True,
):
    """
    一行自动寻找某个目标区域富集的 group。

    例子：
    groups, table = auto_find_enriched_groups(
        adata,
        group_key="niche_leiden",
        target_key="cluster_annotation",
        target_regex="cerebell|cerebellum|CB|Cb",
        section_slices=["C57BL6J-2.060", "C57BL6J-3.016", "C57BL6J-3.004"],
    )
    """

    def _bh_fdr(pvals):
        pvals = np.asarray(pvals, dtype=float)
        order = np.argsort(pvals)
        out = np.empty_like(pvals)
        n = len(pvals)
        prev = 1.0
        for i in range(n - 1, -1, -1):
            rank = i + 1
            val = pvals[order[i]] * n / rank
            prev = min(prev, val)
            out[order[i]] = prev
        return np.clip(out, 0, 1)

    if group_key not in adata.obs:
        raise KeyError(f"{group_key} not found in adata.obs")
    if target_key not in adata.obs:
        raise KeyError(f"{target_key} not found in adata.obs")
    if section_slices is not None and section_key not in adata.obs:
        raise KeyError(f"{section_key} not found in adata.obs")

    group = adata.obs[group_key].astype("string").fillna("NA").astype(str).to_numpy()
    target_label = adata.obs[target_key].astype("string").fillna("NA").astype(str)

    target_mask = np.zeros(adata.n_obs, dtype=bool)

    if target_values is not None:
        target_values = [str(x) for x in target_values]
        target_mask |= target_label.isin(target_values).to_numpy()

    if target_regex is not None:
        target_mask |= target_label.str.contains(
            target_regex,
            case=False,
            regex=True,
            na=False,
        ).to_numpy()

    if target_values is None and target_regex is None:
        raise ValueError("Please provide target_regex or target_values")

    if section_slices is not None:
        section_slices = [str(x) for x in section_slices]
        section_mask = adata.obs[section_key].astype(str).isin(section_slices).to_numpy()
    else:
        section_mask = np.ones(adata.n_obs, dtype=bool)

    background_mask = section_mask
    target_mask = target_mask & section_mask

    n_target_total = int(target_mask.sum())
    n_bg_total = int((background_mask & ~target_mask).sum())

    if n_target_total == 0:
        examples = (
            target_label.value_counts()
            .head(30)
            .rename_axis(target_key)
            .reset_index(name="n_cells")
        )
        raise ValueError(
            f"target_mask selected 0 cells. Check target_regex/target_values. "
            f"Top {target_key} values:\n{examples}"
        )

    rows = []
    for g in sorted(pd.unique(group[background_mask]).astype(str)):
        in_group = background_mask & (group == g)

        n_group = int(in_group.sum())
        n_group_target = int((in_group & target_mask).sum())
        n_group_bg = int((in_group & ~target_mask).sum())

        if n_group < min_group_cells or n_group_target < min_target_cells:
            continue

        frac_group_in_target = n_group_target / max(n_group, 1)
        frac_target_covered = n_group_target / max(n_target_total, 1)
        frac_group_in_background = n_group_bg / max(n_bg_total, 1)
        enrichment = (frac_target_covered + 1e-12) / (frac_group_in_background + 1e-12)

        if fisher_exact is not None:
            table = [
                [n_group_target, n_target_total - n_group_target],
                [n_group_bg, n_bg_total - n_group_bg],
            ]
            _, pval = fisher_exact(table, alternative="greater")
        else:
            pval = np.nan

        rows.append({
            "group": str(g),
            "n_group": n_group,
            "n_group_target": n_group_target,
            "n_group_background": n_group_bg,
            "frac_group_in_target": frac_group_in_target,
            "frac_target_covered": frac_target_covered,
            "enrichment": enrichment,
            "pval": pval,
        })

    table = pd.DataFrame(rows)

    if table.empty:
        if verbose:
            print("No group passed min_group_cells/min_target_cells.")
        return [], table

    if table["pval"].notna().any():
        table["fdr"] = _bh_fdr(table["pval"].fillna(1).to_numpy())
    else:
        table["fdr"] = np.nan

    table = table.sort_values(
        ["enrichment", "frac_group_in_target", "n_group_target"],
        ascending=[False, False, False],
    ).reset_index(drop=True)

    picked = table[
        (table["enrichment"] >= min_enrichment)
        & (table["frac_group_in_target"] >= min_frac_group_in_target)
        & (table["n_group_target"] >= min_target_cells)
    ].copy()

    if use_fdr and picked["fdr"].notna().any():
        picked = picked[picked["fdr"] <= 0.05]

    groups = picked.head(max_groups)["group"].astype(str).tolist()

    if verbose:
        print(f"target cells: {n_target_total:,}")
        print(f"background cells: {int(background_mask.sum()):,}")
        print(f"selected {len(groups)} groups from {group_key}:")
        print(groups)

    return groups, table

In [7]:
major_brain_region_colormap = {'Midbrain': '#FFA6FF',
 'Cerebellum': '#FFFDBC',
 'Isocortex': '#0D9F91',
 'Hippocampus': '#62178d',
 'Cortical_subplate': '#97EC93',
 'Medulla': '#FFA6FF',
 'Olfactory': '#A8ECD3',
 'Fiber_tracts': '#dcb67b',
 'Thalamus': '#FF909F',
 'Hypothalamus': '#F2483B',
 'Pallidum': '#B3C0DF',
 'Pons': '#FFA6FF',
 'Striatum': '#80C0E2',
 'Ventricular_systems': '#AAAAAA',
 'n/a': '#8fec63'}

In [8]:
adata = adata[
    adata.obs["major_brain_region"].notna()
    & (adata.obs["major_brain_region"] != "n/a")
].copy()

In [9]:
niche_leiden_palette, leiden_author_map, confusion = build_leiden_palette_from_author(
    adata,
    leiden_key="niche_leiden",
    author_key="major_brain_region",
    class_palette=major_brain_region_colormap,
    min_lighten=0.20,
    max_lighten=0.50,
    max_hue_shift=2.00,
    min_sat_scale=0.50,
    max_sat_scale=100.00
)

In [10]:
leiden_author_map.to_csv('/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/leiden_author_map.csv')

In [11]:
section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None
                    # groups = ob_cells
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/all/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None
                    # groups = ob_cells
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/all/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

In [12]:
section_slices = ['C57BL6J-2.060', 'C57BL6J-3.016', 'C57BL6J-3.004']

cb_niche_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="niche_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Cerebellum",
    section_slices=section_slices,
)
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Cerebellum'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/cb_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_niche_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/cb_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 47,168
background cells: 244,267
selected 9 groups from niche_leiden:
['57', '76', '32', '98', '94', '101', '46', '35', '26']


In [13]:
section_slices = ['C57BL6J-1.078', 'C57BL6J-3.012']

cb_niche_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="niche_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Thalamus",
    section_slices=section_slices,
)
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Thalamus'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Thalamus_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_niche_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Thalamus_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 15,343
background cells: 135,566
selected 13 groups from niche_leiden:
['96', '51', '25', '70', '31', '50', '38', '71', '49', '4', '13', '1', '104']


In [14]:
section_slices = ['C57BL6J-2.055', 'C57BL6J-1.117']

cb_niche_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="niche_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Midbrain",
    section_slices=section_slices,
)
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Midbrain'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Midbrain_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_niche_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Midbrain_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 15,526
background cells: 47,791
selected 5 groups from niche_leiden:
['34', '39', '53', '3', '73']


In [15]:
section_slices = ['C57BL6J-1.092', 'C57BL6J-3.009']

cb_niche_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="niche_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Fiber_tracts",
    section_slices=section_slices,
)
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Fiber_tracts'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Fiber_tracts_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_niche_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Fiber_tracts_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 16,375
background cells: 151,512
selected 17 groups from niche_leiden:
['9', '24', '8', '46', '43', '86', '6', '22', '84', '69', '3', '66', '73', '48', '26', '104', '97']


In [16]:
section_slices = ['C57BL6J-1.099', 'C57BL6J-1.080', 'C57BL6J-3.008']

cb_niche_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="niche_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Hippocampus",
    section_slices=section_slices,
)
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Hippocampus'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Hippocampus_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_niche_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/Hippocampus_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 19,440
background cells: 210,267
selected 10 groups from niche_leiden:
['90', '55', '45', '63', '54', '16', '0', '2', '41', '42']


In [17]:
section_slices = ['C57BL6J-1.080', 'C57BL6J-3.007', 'C57BL6J-3.015']

cb_niche_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="niche_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Isocortex",
    section_slices=section_slices,
)
for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Isocortex'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/ctx_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_niche_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/ctx_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 54,581
background cells: 274,120
selected 18 groups from niche_leiden:
['23', '102', '91', '80', '68', '59', '56', '78', '74', '52', '20', '89', '29', '60', '61', '28', '48', '40']


In [18]:
section_slices = ['C57BL6J-1.022', 'C57BL6J-1.030', 'C57BL6J-3.006']

cb_niche_groups, cb_niche_table = auto_find_enriched_groups(
    adata,
    group_key="niche_leiden",
    target_key="major_brain_region",   # 改成你的脑区列
    target_regex="Olfactory",
    section_slices=section_slices,
)

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'major_brain_region', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = major_brain_region_colormap, frameon = False, title = section_slice, legend_loc = None,
                    groups = ['Olfactory'],
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/ob_cells/author_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

for section_slice in section_slices:
    temp = adata[adata.obs['brain_section_label'] == section_slice].copy()
    plot = sc.pl.embedding(temp, color = 'niche_leiden', basis = 'X_spatial_coords', 
                    show=False, s=7, palette = niche_leiden_palette, frameon = False, title = section_slice, legend_loc = None,
                    groups = cb_niche_groups
                          )
    scalebar = ScaleBar(0.001, "mm", fixed_value=0.5, location = 'lower left', frameon = False,)
    plot.add_artist(scalebar)
    plot.set_aspect('equal')
    plt.savefig(f"/home/share/huadjyin/home/zhoutao3/tracks/stereoTrack/out/05_merfish_mouseBrain_mae_v1_3axes_train/figures_out/spatial_niche/ob_cells/leiden_{section_slice}.png", 
            bbox_inches="tight",
            transparent=True,
            facecolor='none',
            dpi = 450)
    plt.close()

target cells: 45,535
background cells: 157,306
selected 11 groups from niche_leiden:
['65', '67', '21', '47', '77', '19', '64', '10', '36', '69', '43']


In [19]:
niche_leiden_palette

{'57': '#fffdc9',
 '76': '#cccdff',
 '32': '#ced0ff',
 '97': '#fffed1',
 '98': '#fffed3',
 '101': '#d6d7ff',
 '94': '#d8daff',
 '46': '#fffedb',
 '26': '#f7f6e6',
 '9': '#ffcb79',
 '24': '#ffd086',
 '8': '#ffd492',
 '3': '#ffd99f',
 '86': '#e1d8c9',
 '90': '#8633b6',
 '55': '#6aba3d',
 '54': '#71bd46',
 '63': '#9750c0',
 '45': '#9d59c4',
 '41': '#87c763',
 '42': '#8fcb6c',
 '0': '#ae76ce',
 '17': '#ac9ab7',
 '96': '#ff6458',
 '71': '#60f4ff',
 '103': '#67f4ff',
 '31': '#ff7a6f',
 '33': '#ff8177',
 '83': '#7ff6ff',
 '38': '#87f6ff',
 '75': '#ff978f',
 '4': '#e2b7b4',
 '102': '#33bdaf',
 '91': '#3637be',
 '95': '#53bf3a',
 '74': '#c03db5',
 '78': '#c19e41',
 '23': '#c25b44',
 '80': '#c34875',
 '68': '#a4c54b',
 '89': '#8f4fc6',
 '52': '#52c77c',
 '20': '#5695c8',
 '59': '#5997c9',
 '28': '#5dca84',
 '29': '#9a60cb',
 '60': '#b0cc64',
 '58': '#ce678d',
 '61': '#cf7c6b',
 '56': '#d0b56e',
 '48': '#d172c9',
 '40': '#86d275',
 '2': '#7979d3',
 '6': '#7cd4cc',
 '104': '#98bdb9',
 '79': '#ffb8